# 01 · SQL + EDA

**Goal:** understand the data and build the analysis tables in SQL.

1. Run `python -m src.olist.load_db`, then the files in `sql/` in order (save 01 as `order_facts`, 02 as `customer_facts`).
2. Answer the warm-up questions in `sql/00_explore.sql`.
3. EDA below. For every chart, write one sentence of takeaway under it.

In [ ]:
import sys; sys.path.append('..')   # run from notebooks/ so `src` is importable
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from src.olist.run_sql import query, DB

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

# Fail loudly if setup isn't done, instead of a cryptic SQL error later
assert DB.exists(), f'{DB} missing: download CSVs to data/raw/, then run `python -m src.olist.load_db`'
tables = query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")['name'].tolist()
assert 'order_facts' in tables, 'Build it: python -m src.olist.run_sql sql/01_order_facts.sql --save order_facts'
if 'customer_facts' not in tables:
    print('Note: customer_facts not built yet (sql/02). Fine for now; needed for the Repeat customers section.')
tables

In [ ]:
# Load order_facts and parse dates. One row per delivered order (see sql/01).
order_facts = query('SELECT * FROM order_facts')
for col in ['order_purchase_timestamp', 'order_month']:
    order_facts[col] = pd.to_datetime(order_facts[col])

# Sanity checks: the grain claim in sql/01 should hold
assert order_facts['order_id'].is_unique, 'order_facts should be 1 row per order'
print(order_facts.shape)
print(order_facts.isna().sum()[lambda s: s > 0])   # which columns have gaps, and how many?
order_facts.head()

## Orders over time
Monthly delivered orders. Note any partial months at the edges.

In [ ]:
# Worked example: monthly delivered orders.
monthly = (order_facts
           .groupby('order_month')
           .agg(orders=('order_id', 'size'),
                days_covered=('order_purchase_timestamp', lambda s: s.dt.normalize().nunique()))
           .reset_index())

# Flag partial months: a month with far fewer purchase days than calendar days is incomplete data,
# not a real drop in demand. Don't let it fool a trend line or (later) the forecast.
monthly['calendar_days'] = monthly['order_month'].dt.days_in_month
monthly['partial'] = monthly['days_covered'] < 0.8 * monthly['calendar_days']
display(monthly[monthly['partial']])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly['order_month'], monthly['orders'], marker='o', color='0.6', label='all months')
full = monthly[~monthly['partial']]
ax.plot(full['order_month'], full['orders'], marker='o', label='complete months')
ax.set(title='Delivered orders per month', xlabel=None, ylabel='orders')
ax.legend()
plt.show()

**Takeaway:** _TODO: one sentence, written after you've seen the chart. Which months are partial, and what's the trend over the complete months?_

## Delivery performance
Distribution of `delivery_days` and `days_late`; share of late orders by month and by state.

In [ ]:
# TODO (you): delivery performance
# Hints:
# - delivery_days and days_late are fractional days. Clip or bin for histograms; check the long tail with .describe(percentiles=[.5,.9,.99])
# - Late share by month: order_facts.groupby('order_month')['is_late'].mean(). Mean of a 0/1 column = a rate.
# - By state: also show n per state. A 40% late rate on 30 orders isn't a finding.
# - Think: sql/01 defines is_late as delivered_date > estimated_date. Estimated dates have no time (midnight).
#   What happens to an order delivered ON the estimated day at 3pm? Is that "late"? Decide, and justify it.

## Reviews
Review score distribution; review score vs is_late (preview of notebook 02).

In [ ]:
# TODO (you): reviews
# Hints:
# - review_score value_counts(normalize=True). Note the shape: is it a bell curve?
# - Score by is_late: groupby + mean, AND the share of 1-star reviews. Means hide bimodal distributions.
# - Missing review_score: how many, and are they random? (Compare is_late rate for missing vs. non-missing.)
# - This is a correlation, not a causal effect yet. Notebook 02 handles confounders (state, distance, category).

## Repeat customers
Repeat rate from `customer_facts`; cohort retention heatmap from `sql/03`.

In [ ]:
# TODO (you): repeat customers. Needs sql/02 (customer_facts) and sql/03 (cohort retention) done first.
# Hints:
# - Repeat rate = share of customer_unique_id with n_orders >= 2. Then check: is it ~0? You used customer_id by mistake.
# - Right-censoring: customers whose first order was in the last few months had little time to come back.
#   How would you make the repeat rate fair across cohorts? (e.g., "repeat within 90 days", only cohorts with 90+ days of follow-up)
# - Heatmap: retention.pivot(index='first_month', columns='month_offset', values='retention_rate'), then sns.heatmap.
#   Drop offset 0 (always 100%) or the colour scale gets swamped.

## Takeaways
- 
- 
- 